In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib

# Load data
df = pd.read_csv('/Users/claranatalies/Documents/year 4/COMP 3520 DS/PROJECT/DATASET/base.csv')
target = df['fraud_bool']
features = df.drop('fraud_bool', axis=1)

# Column definitions based on BAF dataset
numerical_cols = [
    'income', 'name_email_similarity', 'prev_address_months_count',
    'current_address_months_count', 'customer_age', 'days_since_request',
    'intended_balcon_amount', 'zip_count_4w', 'velocity_6h', 'velocity_24h',
    'velocity_4w', 'bank_branch_count_8w', 'date_of_birth_distinct_emails_4w',
    'credit_risk_score', 'bank_months_count', 'proposed_credit_limit',
    'session_length_in_minutes', 'device_distinct_emails_8w', 'device_fraud_count',
    'month'
]

categorical_cols = [
    'payment_type', 'employment_status', 'email_is_free', 'housing_status',
    'phone_home_valid', 'phone_mobile_valid', 'has_other_cards', 'foreign_request',
    'source', 'device_os', 'keep_alive_session'
]

# 1. Handle missing values (-1) in numerical columns
features[numerical_cols] = features[numerical_cols].replace(-1, np.nan)
medians = features[numerical_cols].median()
features[numerical_cols] = features[numerical_cols].fillna(medians)

# 2. One-hot encode categoricals
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_cats = encoder.fit_transform(features[categorical_cols])
cat_feature_names = encoder.get_feature_names_out(categorical_cols)
df_cats = pd.DataFrame(encoded_cats, columns=cat_feature_names, index=features.index)

# Drop original categoricals and concatenate
features = features.drop(columns=categorical_cols)
features = pd.concat([features, df_cats], axis=1)

# Save final feature column order (important!)
feature_cols = features.columns.tolist()

# 3. Scale all features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# 4. Save preprocessors
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(encoder, 'encoder.pkl')
joblib.dump(medians, 'medians.pkl')
joblib.dump(feature_cols, 'feature_cols.pkl')

print(f"✅ Preprocessing complete. Feature vector dimension: {scaled_features.shape[1]}")
print("Update your PostgreSQL table's VECTOR dimension to this value if needed.")

✅ Preprocessing complete. Feature vector dimension: 58
Update your PostgreSQL table's VECTOR dimension to this value if needed.


In [ ]:
import numpy as np
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values, Json
import joblib

# Load preprocessors
scaler = joblib.load('scaler.' \
'')
encoder = joblib.load('encoder.pkl')
medians = joblib.load('medians.pkl')
feature_cols = joblib.load('feature_cols.pkl')

# Load original data
df = pd.read_csv('/Users/claranatalies/Documents/year 4/COMP 3520 DS/PROJECT/DATASET/base.csv')
target = df['fraud_bool']
features = df.drop('fraud_bool', axis=1)

# Column definitions (same as preprocess.py)
numerical_cols = [
    'income', 'name_email_similarity', 'prev_address_months_count',
    'current_address_months_count', 'customer_age', 'days_since_request',
    'intended_balcon_amount', 'zip_count_4w', 'velocity_6h', 'velocity_24h',
    'velocity_4w', 'bank_branch_count_8w', 'date_of_birth_distinct_emails_4w',
    'credit_risk_score', 'bank_months_count', 'proposed_credit_limit',
    'session_length_in_minutes', 'device_distinct_emails_8w', 'device_fraud_count',
    'month'
]

categorical_cols = [
    'payment_type', 'employment_status', 'email_is_free', 'housing_status',
    'phone_home_valid', 'phone_mobile_valid', 'has_other_cards', 'foreign_request',
    'source', 'device_os', 'keep_alive_session'
]

# Handle missing values using saved medians
features[numerical_cols] = features[numerical_cols].replace(-1, np.nan)
features[numerical_cols] = features[numerical_cols].fillna(medians)

# One-hot encode categoricals
encoded_cats = encoder.transform(features[categorical_cols])
cat_feature_names = encoder.get_feature_names_out(categorical_cols)
df_cats = pd.DataFrame(encoded_cats, columns=cat_feature_names, index=features.index)

# Drop original categoricals and concatenate
features = features.drop(columns=categorical_cols)
features = pd.concat([features, df_cats], axis=1)

# Ensure column order matches training
features = features[feature_cols]

# Scale - use .values to avoid feature name validation
scaled_features = scaler.transform(features.values)

# Connect to PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    port=5433,
    database="postgres",
    user="postgres",
    password="mysecretpassword"
)
cur = conn.cursor()

# Prepare rows for insertion
rows = []
for idx in range(len(df)):
    vec = scaled_features[idx].tolist()
    metadata = df.iloc[idx].drop('fraud_bool').to_dict()
    rows.append((
        int(target.iloc[idx]),
        int(df.iloc[idx]['month']),
        vec,
        Json(metadata)
    ))

# Batch insert
print("⏳ Inserting data... (may take several minutes)")
execute_values(cur, """
    INSERT INTO applications (fraud_bool, month, feature_vector, metadata)
    VALUES %s
""", rows, page_size=1000)
conn.commit()
cur.close()
conn.close()
print("✅ Data insertion complete.")

/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


⏳ Inserting data... (may take several minutes)
✅ Data insertion complete.
